In [ ]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = 'retina'

import sys
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

notebook_dir = Path().resolve()
project_root = notebook_dir.parent
src_path = str(project_root / "src")

if src_path not in sys.path:
    sys.path.append(src_path)

from hep_tracking.features import create_pair_dataset, split_by_event
from hep_tracking.classifiers import train_random_forest, train_xgboost, train_lightgbm, evaluate_classifier_throughput
from hep_tracking.plots import plot_roc_curves, plot_pr_curves
from hep_tracking.utils import calculate_classification_metrics

print(f"Katalog główny projektu: {project_root}")
print("Wszystkie moduły załadowane pomyślnie.")

In [ ]:
data_dir = project_root / "data"
dataset_name = "dataset_hard_1M.npz"
candidates_name = "candidates_hard_1M.npz"

data_path = data_dir / dataset_name
candidates_path = data_dir / candidates_name

if not data_path.exists() or not candidates_path.exists():
    raise FileNotFoundError("Brak plików danych. Upewnij się, że wygenerowano dane i kandydatów.")

loaded_data = np.load(data_path)
features_5d = loaded_data["X"]
labels = loaded_data["y"]
event_ids = loaded_data["event_id"]

loaded_candidates = np.load(candidates_path)
candidate_indices = loaded_candidates["indices"]

print(f"=== Podsumowanie wczytanych danych surowych ===")
print(f"Plik danych:       {dataset_name}")
print(f"Plik kandydatów:   {candidates_name}")
print(f"Liczba hitów (N):  {features_5d.shape[0]:,}")
print(f"Wymiar cech hitu:  {features_5d.shape[1]}")
print(f"Liczba zdarzeń:    {len(np.unique(event_ids))}")
print(f"Liczba sąsiadów:   {candidate_indices.shape[1]}")

In [ ]:
print("=== Feature Engineering: Tworzenie zbioru par ===")
start_time = time.perf_counter()

X_pairs, y_pairs, event_ids_pairs = create_pair_dataset(
    features=features_5d,
    labels=labels,
    event_ids=event_ids,
    candidate_indices=candidate_indices,
    max_neg_ratio=5.0,
    seed=42
)

prep_time = time.perf_counter() - start_time

n_total = len(y_pairs)
n_pos = np.sum(y_pairs)
n_neg = n_total - n_pos

print(f"Czas inżynierii cech: {prep_time:.2f} s")
print(f"Liczba par (M):       {n_total:,}")
print(f"Wymiar cech pary:     {X_pairs.shape[1]}")
print(f"Pary pozytywne:       {n_pos:,} ({n_pos/n_total*100:.1f}%)")
print(f"Pary negatywne:       {n_neg:,} ({n_neg/n_total*100:.1f}%)")

In [ ]:
print("=== Podział na zbiory Train / Val / Test ===")

X_train, y_train, X_val, y_val, X_test, y_test = split_by_event(
    X=X_pairs,
    y=y_pairs,
    event_ids=event_ids_pairs,
    train_size=0.75,
    val_size=0.15,
    seed=42
)

print(f"Rozmiar macierzy treningowej:  {X_train.shape[0]:,} par")
print(f"Rozmiar macierzy walidacyjnej: {X_val.shape[0]:,} par")
print(f"Rozmiar macierzy testowej:     {X_test.shape[0]:,} par")
print("Podział zakończony sukcesem. Maski logiczne zapewniają brak przecieku zdarzeń.")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
import lightgbm as lgb
from hep_tracking.classifiers import optimize_hyperparameters

print("=== Optymalizacja Hiperparametrów i Trening Modeli ===")

search_size = min(50000, len(X_train))
rng = np.random.default_rng(42)
search_indices_train = rng.choice(len(X_train), search_size, replace=False)
search_indices_val = rng.choice(len(X_val), int(search_size * 0.2), replace=False)

X_train_sub = X_train[search_indices_train]
y_train_sub = y_train[search_indices_train]
X_val_sub = X_val[search_indices_val]
y_val_sub = y_val[search_indices_val]

trained_models = {}
training_times = {}

print("\n--- Random Forest ---")
rf_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [10, 15, 20, None],
    "min_samples_split": [2, 5, 10]
}
rf_base = RandomForestClassifier(random_state=42)
print("Szukanie hiperparametrów...")
rf_best = optimize_hyperparameters(rf_base, rf_grid, X_train_sub, y_train_sub, X_val_sub, y_val_sub, n_iter=5)
print(f"Najlepsze parametry: {rf_best}")

start_rf = time.perf_counter()
trained_models["Random Forest"] = train_random_forest(
    X_train, y_train, 
    n_jobs=-1, 
    random_state=42,
    **rf_best
)
training_times["Random Forest"] = time.perf_counter() - start_rf
print(f"Trening zakończony w {training_times['Random Forest']:.2f} s")

print("\n--- XGBoost ---")
xgb_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [4, 6, 8],
    "learning_rate": [0.01, 0.05, 0.1]
}
xgb_base = xgb.XGBClassifier(random_state=42, eval_metric="logloss")
print("Szukanie hiperparametrów...")
xgb_best = optimize_hyperparameters(xgb_base, xgb_grid, X_train_sub, y_train_sub, X_val_sub, y_val_sub, n_iter=5)
print(f"Najlepsze parametry: {xgb_best}")

start_xgb = time.perf_counter()
trained_models["XGBoost"] = train_xgboost(
    X_train, y_train, X_val, y_val,
    early_stopping_rounds=15,
    n_jobs=-1,
    random_state=42,
    **xgb_best
)
training_times["XGBoost"] = time.perf_counter() - start_xgb
print(f"Trening zakończony w {training_times['XGBoost']:.2f} s")

print("\n--- LightGBM ---")
lgb_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [4, 6, 8, -1],
    "learning_rate": [0.01, 0.05, 0.1]
}
lgb_base = lgb.LGBMClassifier(random_state=42, verbose=-1)
print("Szukanie hiperparametrów...")
lgb_best = optimize_hyperparameters(lgb_base, lgb_grid, X_train_sub, y_train_sub, X_val_sub, y_val_sub, n_iter=5)
print(f"Najlepsze parametry: {lgb_best}")

start_lgb = time.perf_counter()
trained_models["LightGBM"] = train_lightgbm(
    X_train, y_train, X_val, y_val,
    n_jobs=-1,
    random_state=42,
    **lgb_best
)
training_times["LightGBM"] = time.perf_counter() - start_lgb
print(f"Trening zakończony w {training_times['LightGBM']:.2f} s")

pd.DataFrame(list(training_times.items()), columns=["Model", "Czas treningu [s]"]).round(2)

In [ ]:
print("=== Ewaluacja jakości klasyfikacji (Zbiór Testowy) ===")

metrics_list = []

for model_name, model in trained_models.items():
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    metrics = calculate_classification_metrics(y_test, y_pred_proba)
    metrics["Model"] = model_name
    
    metrics_list.append(metrics)

df_metrics = pd.DataFrame(metrics_list)
df_metrics.set_index("Model", inplace=True)

df_metrics.round(4)

In [ ]:
print("=== Wykresy (Deliverables) ===")

plot_roc_curves(trained_models, X_test, y_test)

plot_pr_curves(trained_models, X_test, y_test)

In [ ]:
print("=== Analiza przepustowości (Throughput / Latency) ===")

throughput_results = []
batch_sizes_to_test = [1, 1000, 10000]

for model_name, model in trained_models.items():
    results = evaluate_classifier_throughput(
        model=model, 
        X_test=X_test, 
        batch_sizes=batch_sizes_to_test, 
        num_runs=5
    )
    
    for batch_size, throughput in results.items():
        throughput_results.append({
            "Model": model_name,
            "Rozmiar wsadu (Batch)": batch_size,
            "Przepustowość (pary/s)": throughput
        })

df_throughput = pd.DataFrame(throughput_results)

df_pivot = df_throughput.pivot(
    index="Model", 
    columns="Rozmiar wsadu (Batch)", 
    values="Przepustowość (pary/s)"
)

df_pivot_formatted = df_pivot.map(lambda x: f"{x:,.0f}")
df_pivot_formatted